## OpenHackathon Technical Intro: Single‑GPU NIMs (MolMIM, DiffDock) and Workflow Tips

This guide shows how to:
- Launch relevant NVIDIA NIMs on a single node with a single GPU
- Obtain and configure your `NGC_API_KEY`
- Use the included pre‑fingerprinted ChEMBL database for rapid novelty checks
- Run a minimal iterative molecule generation loop inspired by MolMIM oracle‑controlled generation

Reference for MolMIM workflow: [MolMIM Oracle‑Controlled Generation notebook](https://github.com/NVIDIA/bionemo-examples/blob/main/examples/nims/molmim/MolMIMOracleControlledGeneration.ipynb).


### Prerequisites (single node, single GPU)

- NVIDIA GPU with recent drivers (e.g., CUDA‑compatible; check with `nvidia-smi`).
- Docker Engine (24+ recommended) and NVIDIA Container Toolkit installed and working.
- Outbound network access to pull NIM images from `nvcr.io`.
- Sufficient GPU/host memory (suggested: 12–24GB GPU, 16–32GB RAM).


### 1) Get and set your NVIDIA NGC API key

1. Create an NGC account (free) and generate an API key.
   - Visit NGC: `https://ngc.nvidia.com/`
   - Sign in → User menu → Setup → API Key → Generate Key
2. Store the key as an environment variable on your node:
```bash
export NGC_API_KEY="<your-key-here>"
# Optional: persist it to your shell profile
echo 'export NGC_API_KEY="<your-key-here>"' >> ~/.bashrc
source ~/.bashrc
```
3. Verify access (Docker required):
```bash
docker login nvcr.io --username \$oauthtoken --password $NGC_API_KEY
```


### 2) Launch NIM services on a single GPU

Below are minimal one‑liner commands to launch each NIM in a single‑GPU setup. Adjust memory as needed.

- MolMIM NIM (example tag; use latest available):
```bash
docker run -d --rm --name molmim-nim \
  --runtime=nvidia --gpus "device=0" \
  -p 9000:8000 -e NGC_API_KEY \
  -v "$HOME/.cache/nim:/opt/nim/.cache" \
  --memory=16g --shm-size=2g \
  nvcr.io/nim/bionemo/molmim:latest
```

- DiffDock NIM (example tag; use latest available):
```bash
docker run -d --rm --name diffdock-nim \
  --runtime=nvidia --gpus "device=0" \
  -p 9001:8000 -e NGC_API_KEY \
  -v "$HOME/.cache/nim:/opt/nim/.cache" \
  --memory=16g --shm-size=2g \
  nvcr.io/nim/mit/diffdock:latest
```

Notes:
- Ensure the GPU is free and drivers/CUDA are installed.
- If ports `9000/9001` are busy, choose different host ports.
- Keep one NIM running at a time if your GPU memory is limited.


### 3) Quick client calls to MolMIM and DiffDock

Once containers are up, you can call them via REST or Python clients. Example using `requests`:

```python
import os, requests

MOLMIM_URL = os.environ.get("MOLMIM_URL", "http://localhost:9000")
DIFFDOCK_URL = os.environ.get("DIFFDOCK_URL", "http://localhost:9001")

# Example: MolMIM generation (payload shape may vary by version)
payload = {
    "num_mols": 8,
    "constraints": {"qed_min": 0.5},
    "seed": 123
}
r = requests.post(f"{MOLMIM_URL}/v1/generate", json=payload, timeout=300)
r.raise_for_status()
molmim_smiles = r.json().get("smiles", [])
print("MolMIM generated:", molmim_smiles[:3])

# Example: DiffDock docking of first molecule against a target protein
if molmim_smiles:
    dock_request = {
        "protein_pdb": "...PDB_CONTENT_OR_URL...",  # supply a CDK4/6/11 structure
        "ligand_smiles": molmim_smiles[0],
        "num_samples": 16
    }
    d = requests.post(f"{DIFFDOCK_URL}/v1/dock", json=dock_request, timeout=600)
    d.raise_for_status()
    pose = d.json()
    print("DiffDock pose keys:", list(pose.keys()))
```

Notes:
- Confirm endpoint paths from container logs or docs (`/v1/generate`, `/v1/dock` may differ by release).
- For larger jobs, batch multiple ligands and cache results.


### 4) Fast novelty checks with pre‑fingerprinted ChEMBL

The folder contains precomputed fingerprints under `openhackathon/chembl_data/` (e.g., `chembl_fingerprints.pkl`) for quick novelty assessment.

Minimal example:
```python
import os, pickle
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

chembl_fp_path = os.path.join("./chembl_data", "chembl_fingerprints.pkl")
with open(chembl_fp_path, "rb") as f:
    chembl_fps = pickle.load(f)  # {chembl_id: RDKit ExplicitBitVect}

# Query compounds (CSV with column `smiles`)
df = pd.DataFrame({"smiles": [
    "CC1=C(C(=O)N(C2=CC=CC=C2)N1C)C3=CC=C(C=C3)Cl",
    "CC1=CC=C(C=C1)C2=CC(=NN2C3=CC=C(C=C3)S(=O)(=O)N)C(F)(F)F"
]})

def morgan_fp(smiles):
    m = Chem.MolFromSmiles(smiles)
    return AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048) if m else None

# Compute max Tanimoto similarity vs ChEMBL per query
max_sims = []
for s in df.smiles:
    qfp = morgan_fp(s)
    if qfp is None:
        max_sims.append(float("nan"))
        continue
    max_sim = 0.0
    for _, ref_fp in chembl_fps.items():
        sim = DataStructs.TanimotoSimilarity(qfp, ref_fp)
        if sim > max_sim:
            max_sim = sim
    max_sims.append(max_sim)

df["max_chembl_similarity"] = max_sims
# Novelty rule of thumb (adjust threshold as needed)
SIM_THRESHOLD = 0.85
df["is_novel"] = df["max_chembl_similarity"] < SIM_THRESHOLD
print(df)
```
Notes:
- This brute‑force loop is fine for small batches; for 1k compounds, consider chunking or approximate search.
- The evaluation scripts include similar logic; you can reuse them for consistency.


### 5) Minimal MolMIM‑style iterative optimization loop (single‑GPU, toy example)

The following sketch adapts ideas from the MolMIM oracle‑controlled generation workflow to a simple greedy loop:

```python
# Pseudocode/toy example: adapt endpoints/fields to your NIM release
import os, random, requests
import pandas as pd
from rdkit import Chem
from rdkit.Chem import QED

MOLMIM_URL = os.environ.get("MOLMIM_URL", "http://localhost:9000")

def qed_score(s):
    m = Chem.MolFromSmiles(s)
    return QED.qed(m) if m else 0.0

def generate_batch(num=64, seed=0):
    payload = {"num_mols": num, "seed": seed}
    r = requests.post(f"{MOLMIM_URL}/v1/generate", json=payload, timeout=600)
    r.raise_for_status()
    return r.json().get("smiles", [])

# Simple objective: maximize QED while enforcing a novelty threshold against ChEMBL
from rdkit.Chem import AllChem, DataStructs
import pickle
chembl_fp_path = os.path.join("./chembl_data", "chembl_fingerprints.pkl")
chembl_fps = pickle.load(open(chembl_fp_path, "rb"))

def max_chembl_sim(s):
    m = Chem.MolFromSmiles(s)
    if not m:
        return 1.0
    qfp = AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048)
    best = 0.0
    for _, ref_fp in chembl_fps.items():
        sim = DataStructs.TanimotoSimilarity(qfp, ref_fp)
        if sim > best:
            best = sim
    return best

NOVELTY_THRESH = 0.85

best = []
random.seed(0)
for it in range(3):  # 3 iterations for demo
    batch = generate_batch(num=64, seed=it)
    scored = []
    for s in batch:
        sim = max_chembl_sim(s)
        if sim < NOVELTY_THRESH:
            scored.append((s, 0.7*qed_score(s) + 0.3*(1.0 - sim)))
    # keep top 20 as seeds for next round (could condition MolMIM with them if supported)
    top = sorted(scored, key=lambda x: x[1], reverse=True)[:20]
    best.extend(top)
    print(f"Iter {it}: kept {len(top)}; current best score={top[0][1]:.3f}")

best_df = pd.DataFrame(best, columns=["smiles", "score"]).drop_duplicates("smiles")
print(best_df.head())
```

Tips:
- Replace the toy objective with a multi‑objective function (e.g., selectivity proxy, toxicity proxy).
- You can use DiffDock evaluations inside the loop to penalize favorable CDK11 poses or reward CDK4/6 poses.
- See the reference MolMIM notebook for more advanced conditioning and oracle strategies:
  - [MolMIM Oracle‑Controlled Generation](https://github.com/NVIDIA/bionemo-examples/blob/main/examples/nims/molmim/MolMIMOracleControlledGeneration.ipynb)


### 6) Practical single‑GPU constraints and tips

- Prefer running a single NIM at a time on 12–24GB GPUs; monitor memory with `nvidia-smi`.
- Batch requests (e.g., 32–128 ligands) and reuse server‑side caches when possible.
- Persist `NGC_API_KEY` and container images between sessions to avoid re‑pull costs.
- For DiffDock, down‑tune sampling if memory/time constrained; for MolMIM, reduce batch sizes.
- Always validate generated SMILES (RDKit) before downstream processing; drop invalids early.
